In [ ]:
import torch

print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

In [ ]:
#Google Drive 연동
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Drive에 있는 데이터 폴더를 직접 사용합니다.
DATA_DIR = Path("/content/drive/MyDrive/SkinCare/data")

train_source = DATA_DIR / "train"
val_source = DATA_DIR / "val"

# 폴더와 클래스 구성이 맞는지 확인합니다.
assert train_source.is_dir(), f"학습 폴더를 찾을 수 없습니다: {train_source}"
assert val_source.is_dir(), f"검증 폴더를 찾을 수 없습니다: {val_source}"

train_classes = {p.name for p in train_source.iterdir() if p.is_dir()}
val_classes = {p.name for p in val_source.iterdir() if p.is_dir()}

assert train_classes, "학습 폴더에 클래스 폴더가 없습니다."
assert train_classes == val_classes, "학습과 검증의 클래스 폴더명이 다릅니다."

print("데이터셋 경로 확인 완료")
print("클래스:", sorted(train_classes))
print("학습 클래스 수:", len(train_classes))

In [12]:
#YOLO 패키지 설치
!pip install -q ultralytics
from ultralytics import YOLO

In [13]:
#하이퍼파라미터 설정
MODEL_NAME = "yolo11n-cls.pt" #yolo분류 사전 학습 가중치
EPOCHS = 10
IMAGE_SIZE = 224
BATCH_SIZE = 32
LEARNING_RATE = 0.001
OPTIMIZER = "auto"
WEIGHT_DECAY = 0.0005
PATIENCE = 10
WORKERS = 2
DEVICE = 0
SEED = 42

In [14]:
train_acc_list = []

def calculate_train_accuracy(trainer):

    model = trainer.model

    # 기존 학습 모드 저장
    was_training = model.training

    model.eval()

    correct = 0
    total = 0

    with torch.inference_mode():

        for images, labels in train_loader:

            images = images.to(trainer.device)
            labels = labels.to(trainer.device)

            outputs = model(images)

            # Tuple에서 실제 분류 결과 Tensor 추출
            if isinstance(outputs, tuple):
              outputs = outputs[0]
              predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total

    train_acc_list.append(accuracy)

    print(
        f"Epoch {trainer.epoch + 1} "
        f"Training Accuracy: {accuracy * 100:.2f}%"
    )

    # 학습 모드 복구
    if was_training:
        model.train()

In [ ]:
# 사전 학습 모델 로드
model = YOLO(MODEL_NAME)

# # Training Accuracy 콜백 등록
# model.add_callback(
#     "on_train_epoch_end",
#     calculate_train_accuracy
# )

# 학습
results = model.train(
    data=DATA_DIR,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    lr0=LEARNING_RATE,
    optimizer=OPTIMIZER,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    workers=WORKERS,
    device=DEVICE,
    seed=SEED,
    project="/content/runs/classify",
    name="skin_classifier",
    task="classify",
)

Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/SkinCare/data, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=None, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=skin_classifier-2, nbs=64, nms=None, opset=Non

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

#학습 결과 CSV 경로
RESULT_DIR = "/content/runs/classify/skin_classifier"
CSV_PATH = os.path.join(RESULT_DIR, "results.csv")
BEST_MODEL_PATH = os.path.join(RESULT_DIR, "weights", "best.pt")

#학습 결과 불러오기
df = pd.read_csv(CSV_PATH)

#컬럼명 공백 제거
df.columns = df.columns.str.strip()

#그래프 크기 설정
plt.figure(figsize=(14, 5))

# 정확도 그래프
plt.subplot(1, 2, 1)

# plt.plot(df["epoch"], train_acc_list,
#          label="Training Accuracy", marker="o")

plt.plot(df["epoch"], df["metrics/accuracy_top1"],
         label="Validation Accuracy", marker="o")

plt.plot(df["epoch"], df["metrics/accuracy_top5"],
         label="Validation Top-5 Accuracy", marker="o")

plt.title("YOLO11 Classification Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

#손실 그래프
plt.subplot(1, 2, 2)
plt.plot(df["epoch"], df["train/loss"],
         label="Training Loss", marker="o")
plt.plot(df["epoch"], df["val/loss"],
         label="Validation Loss", marker="o")
plt.title("YOLO11 Classification Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/classify/skin_classifier/results.csv'